<a href="https://colab.research.google.com/github/sruthisahu/indivulntnbc/blob/main/day1_setup_ipnyb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [45]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [46]:
!git clone https://github.com/sruthisahu/indivulntnbc.git
%cd indivulntnbc
!git config --global user.email "sruthisahu836@gmail.com"
!git config --global user.name "sruthisahu"
!mkdir -p data/raw data/processed notebooks src figures docs

fatal: destination path 'indivulntnbc' already exists and is not an empty directory.
[Errno 20] Not a directory: 'indivulntnbc'
/content/indivulntnbc


In [47]:
!pip install cobra xgboost shap networkx transformers pandas numpy requests mygene biopython -q
!pip freeze | grep -E "cobra|cobra|xgboost|shap|networkx|transformers|pandas|numpy|requests|mygene|biopython" > /content/indivulntnbc/docs/requirements_day1.txt

In [48]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

CUDA available: False
Device: CPU only


In [49]:
import os, shutil, pandas as pd
from google.colab import drive



os.makedirs('/content/drive/MyDrive/indivulntnbc-data/', exist_ok=True)

for fname in ['CRISPRGeneEffect.csv', 'Model.csv']:
    src = f'/content/{fname}'
    dst = f'/content/drive/MyDrive/indivulntnbc-data/{fname}'

    if os.path.exists(src):

        shutil.copy(src, dst)
        print(f"Copied {fname} safely to Drive")
    elif os.path.exists(dst):
        print(f"{fname} already in Drive, skipping")
    else:
        print(f"WARNING: {fname} not found - check upload")


crispr = pd.read_csv('/content/drive/MyDrive/indivulntnbc-data/CRISPRGeneEffect.csv')

crispr.set_index(crispr.columns[0], inplace=True)

models = pd.read_csv('/content/drive/MyDrive/indivulntnbc-data/Model.csv')
print(crispr.shape, models.shape)

CRISPRGeneEffect.csv already in Drive, skipping
Model.csv already in Drive, skipping
(61, 6260) (2154, 49)


In [50]:
breast_lines = models[models['OncotreeLineage']== 'Breast']
print(breast_lines.shape)
print(breast_lines['OncotreeSubtype'].value_counts())

(96, 49)
OncotreeSubtype
Breast Invasive Ductal Carcinoma     40
Invasive Breast Carcinoma            22
Breast Invasive Lobular Carcinoma    14
Breast Invasive Carcinoma, NOS        5
Breast Ductal Carcinoma In Situ       5
Immortalized Breast Cells             4
Breast Neoplasm, NOS                  4
Breast Invasive Cancer, NOS           2
Name: count, dtype: int64


In [51]:
clinical = pd.read_csv('/content/brca_tcga_clinical_data.tsv', sep='\t')
print(clinical.columns.tolist())

['studyId', 'patientId', 'sampleId', 'CANCER_TYPE', 'CANCER_TYPE_DETAILED', 'DAYS_TO_COLLECTION', 'FRACTION_GENOME_ALTERED', 'IS_FFPE', 'MUTATION_COUNT', 'OCT_EMBEDDED', 'ONCOTREE_CODE', 'OTHER_SAMPLE_ID', 'PATHOLOGY_REPORT_FILE_NAME', 'PATHOLOGY_REPORT_UUID', 'SAMPLE_INITIAL_WEIGHT', 'SAMPLE_TYPE', 'SAMPLE_TYPE_ID', 'SOMATIC_STATUS', 'TMB_NONSYNONYMOUS', 'VIAL_NUMBER', 'AGE', 'AJCC_METASTASIS_PATHOLOGIC_PM', 'AJCC_NODES_PATHOLOGIC_PN', 'AJCC_PATHOLOGIC_TUMOR_STAGE', 'AJCC_STAGING_EDITION', 'AJCC_TUMOR_PATHOLOGIC_PT', 'DAYS_TO_INITIAL_PATHOLOGIC_DIAGNOSIS', 'ER_STATUS_BY_IHC', 'ER_STATUS_IHC_PERCENT_POSITIVE', 'ETHNICITY', 'FORM_COMPLETION_DATE', 'HER2_CENT17_RATIO', 'HER2_FISH_STATUS', 'HER2_IHC_SCORE', 'HISTOLOGICAL_DIAGNOSIS', 'HISTORY_NEOADJUVANT_TRTYN', 'HISTORY_OTHER_MALIGNANCY', 'ICD_10', 'ICD_O_3_HISTOLOGY', 'ICD_O_3_SITE', 'IHC_HER2', 'INFORMED_CONSENT_VERIFIED', 'INITIAL_PATHOLOGIC_DX_YEAR', 'LYMPH_NODES_EXAMINED', 'LYMPH_NODE_EXAMINED_COUNT', 'MENOPAUSE_STATUS', 'METHOD_OF_I

In [52]:
%cd /content/indivulntnbc
!rm- rf Human-GEM
!git clone https://github.com/SysBioChalmers/Human-GEM.git
import cobra
model = cobra.io.read_sbml_model('Human-GEM/model/Human-GEM.xml')
print(len(model.genes), len(model.reactions), len(model.metabolites))
print(model.genes[0].id)

/content/indivulntnbc
/bin/bash: line 1: rm-: command not found
fatal: destination path 'Human-GEM' already exists and is not an empty directory.
2848 12931 8461
ENSG00000000419


In [53]:
import re
import mygene
import pandas as pd

depmap_genes = crispr.columns.tolist()
parsed = []

for g in depmap_genes:

    match = re.search(r'^(.*?)\s*\((\d+)\)', str(g))
    if match:

        parsed.append(match.groups())
    else:

        print(f"Skipped column (not a gene format): {g}")

depmap_map = pd.DataFrame(parsed, columns=['symbol', 'entrez'])

mg = mygene.MyGeneInfo()
result = mg.querymany(
    depmap_map['symbol'].tolist(),
    scopes='symbol',
    fields='ensembl.gene,entrezgene',
    species='human'
)

mapping_df = pd.DataFrame(result)

mapping_df.to_csv('/content/indivulntnbc/data/processed/gene_id_mapping.csv', index=False)

unmapped_count = mapping_df['notfound'].sum() if 'notfound' in mapping_df.columns else 0
print(f"\n{unmapped_count} genes unmapped")

INFO:biothings.client:Finished.


Skipped column (not a gene format): -0.1254238437396673
Skipped column (not a gene format): -0.7667513416347671
Skipped column (not a gene format): 0.16090032842648724
Skipped column (not a gene format): -0.15126837687829042
Skipped column (not a gene format): -0.08032944390685567
Skipped column (not a gene format): -0.07536984280190881
Skipped column (not a gene format): 0.02916958889035823
Skipped column (not a gene format): -0.061656542738961595
Skipped column (not a gene format): -0.3011737388508072
Skipped column (not a gene format): 0.08881093148586904
Skipped column (not a gene format): -0.04665719210991066
Skipped column (not a gene format): 0.0211884546003646
Skipped column (not a gene format): 0.027312024507216103
Skipped column (not a gene format): -0.17833024766415484
Skipped column (not a gene format): -0.01263596170439854
Skipped column (not a gene format): 0.15814749721203666
Skipped column (not a gene format): 0.0398847991919081
Skipped column (not a gene format): -0.01

In [54]:
print(crispr.isnull().sum().sum(), "missing values in CRISPR matrix")
print(crispr.index.duplicated().sum(), "duplicate cell lines")
print(clinical.duplicated().sum(), "duplicate patient rows")
print(f"{crispr.shape[0]} cell lines, {crispr.shape[1]} genes")
print(f"{clinical.shape[0]} patients in TCGA-BRCA")

tnbc_mask = (
    (clinical['ER_STATUS_BY_IHC'] == 'Negative')&
    (clinical['PR_STATUS_BY_IHC'] == 'Negative')&
    (clinical['IHC_HER2'] == 'Negative')
)
print(f"{tnbc_mask.sum()} TNBC patients identified")

25612 missing values in CRISPR matrix
0 duplicate cell lines
0 duplicate patient rows
61 cell lines, 6260 genes
500 patients in TCGA-BRCA
53 TNBC patients identified


In [55]:
%cd /content/indivulntnbc/
!git add .
!git commit -m "Day 1: environment, raw data, gene ID mapping ,QC, bibliography"
!git push

/content/indivulntnbc
hint: You've added another git repository inside your current repository.
hint: Clones of the outer repository will not contain the contents of
hint: the embedded repository and will not know how to obtain it.
hint: If you meant to add a submodule, use:
hint: 
hint: 	git submodule add <url> Human-GEM
hint: 
hint: If you added this path by mistake, you can remove it from the
hint: index with:
hint: 
hint: 	git rm --cached Human-GEM
hint: 
hint: See "git help submodule" for more information.
[main 2f2f8ef] Day 1: environment, raw data, gene ID mapping ,QC, bibliography
 4 files changed, 522 insertions(+)
 create mode 160000 Human-GEM
 create mode 100644 brca_tcga_clinical_data.tsv
 create mode 100644 data/processed/gene_id_mapping.csv
 create mode 100644 docs/requirements_day1.txt
fatal: could not read Username for 'https://github.com': No such device or address
